# Mask Calculation

In [ ]:
import os
import itertools
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import tifffile
import matplotlib.pyplot as plt
from PIL import Image
import geopandas as gpd

import warnings
warnings.filterwarnings("ignore")

In [ ]:
def mask2shapely(masks_in):
    from shapely.geometry import Polygon
    import geopandas as gpd
    from skimage import measure
    import numpy as np
    
    ids = []
    geometries = []

    for i in np.unique(masks_in)[1:]:
        binary_mask = (masks_in == i).astype(np.uint8)
        
        # Find contours using scikit-image
        contours = measure.find_contours(binary_mask, 0.5)
        
        # If multiple contours (disconnected pixels), take the longest one
        if len(contours) > 1:
            longest_contour = max(contours, key=len)
            # Convert from (row, col) to (x, y) coordinates
            coords = longest_contour[:, [1, 0]]
            if len(coords) > 3:
                geometries.append(Polygon(coords))
                ids.append(i)
        else:
            if len(contours) > 0 and len(contours[0]) > 3:
                # Convert from (row, col) to (x, y) coordinates
                coords = contours[0][:, [1, 0]]
                geometries.append(Polygon(coords))
                ids.append(i)
    
    gdf = gpd.GeoDataFrame({'id': ids, 'geometry': geometries}, crs='EPSG:4326')
    gdf = gdf[gdf['id'] != 0]
    #not needed here
    #gdf['x'] = gdf.geometry.centroid.x
    #gdf['y'] = gdf.geometry.centroid.y   

    return gdf

In [ ]:
# BM ROI

md = pd.read_csv("/mnt/disks/data/imc/CART_cohort/in/sample_md.csv")

from tqdm import tqdm

for sample_id in tqdm(md['sample'].unique()):
        
    #Skip ungated samples
    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    #skip if done, allow new samples
    if 'mask_V1withAdipo_gdf.geojson' in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue

    mask = tifffile.imread(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo.tif')

    gdf = mask2shapely(mask)

    gdf.to_file(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo_gdf.geojson', driver='GeoJSON')

In [ ]:
# EMD ROI

obs_data = sc.read('/mnt/disks/data/imc/CART_cohort/out/emd_work/emd_clustering-all.h5ad').obs

from tqdm import tqdm

sample_ids = [i.split('.')[0] for i in os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/emd_tiffs/masks/")]

for sample_id in tqdm(sample_ids):

    #import
    mask = tifffile.imread(f"/mnt/disks/data/imc/CART_cohort/in/emd_tiffs/masks/{sample_id}.tiff")

    #subset gated
    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample = obs_sample.set_index('ObjectNumber')
    obs_sample = obs_sample[obs_sample.gated]
    shared_cells = np.intersect1d(mask,obs_sample.index)
    mask = np.where(np.isin(mask,shared_cells),mask,0)

    gdf = mask2shapely(mask)

    gdf.to_file(f'/mnt/disks/data/imc/CART_cohort/out/emd_data/masks_gdf/{sample_id}.geojson', driver='GeoJSON')